# Run analysis — plots per run, leak vs no-leak

Group 32 · Aircraft Hydraulic Leak Detection

1. **One run:** pressure, flow, flow loss and acoustic loudness against time.
2. **Leak vs no-leak:** the key window features compared between classes, which PZT hears each leak
   loudest (localisation), whether the flow loss matches the jug-measured leak rate, and which features
   separate leak from no-leak best.

Every comparison uses **one value per run** (the median over that run's 1 s windows). Windows from the
same run are almost copies of each other, so counting them separately would make differences look more
certain than they are.

**Before you start:** run `python check_runs.py` and fix any FAIL. Open this notebook from the `python/`
folder (the paths are relative). Figures are saved to `results/figures/`.

In [ ]:
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import CLASS_NAMES
from features import build_dataset

RAW_DIR = "data/raw"
FIG_DIR = "results/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Colours: fixed order, one colour per class / sensor everywhere in this notebook
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
SHORT = {0: "No leak", 1: "L1 centre", 2: "L2 right", 3: "L3 left"}
CLASS_COLOR = dict(zip(SHORT, SLOTS))

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.5, "legend.frameon": False,
})

# Run list from the JSON label files
runs = []
for js in sorted(glob.glob(os.path.join(RAW_DIR, "run_*.json"))):
    m = json.load(open(js))
    csv = js[:-5] + ".csv"
    rows = sum(1 for _ in open(csv)) - 1 if os.path.exists(csv) else 0
    runs.append(dict(run_id=m["run_id"], label=m["label"], cls=SHORT.get(m["label"], "?"),
                     leak_lpm=m.get("leak_lpm", 0.0), seconds=rows / 10,
                     condition=m.get("condition", ""), file=csv))
runs = pd.DataFrame(runs).set_index("run_id")
SIMULATED = runs.condition.str.contains("SIMULATED").any()
if SIMULATED:
    print("WARNING: SIMULATED data. These plots only test the notebook. Never put them in the report.")
print(f"{len(runs)} runs:", runs.cls.value_counts().to_dict())
runs.drop(columns="file")

In [ ]:
def mark_simulated(fig):
    # Watermark so a simulated figure can never be mistaken for a real result
    if SIMULATED:
        fig.text(0.5, 0.5, "SIMULATED DATA", ha="center", va="center", rotation=20,
                 fontsize=44, color=MUTED, alpha=0.18, weight="bold", zorder=0)

def save(fig, name):
    path = os.path.join(FIG_DIR, ("SIM_" if SIMULATED else "") + name)
    fig.savefig(path, bbox_inches="tight")
    print("saved", path)

## 1 · One run against time

Change `RUN` and re-run the cell. What to look for:
- **Pressure:** drops when a leak opens; steps when the pump speed changes.
- **Flow loss Q1 − Q2:** about 0 with no leak (after the `N` null). With a leak it should be close to
  the jug-measured rate.
- **AE loudness:** the PZT nearest the leak should be loudest. A flat line or a line stuck at 0 is a
  sensor or STM32 fault.

In [ ]:
RUN = 2

def plot_run(run_id):
    r = runs.loc[run_id]
    df = pd.read_csv(r.file).apply(pd.to_numeric, errors="coerce").dropna()
    t = (df.t_ms - df.t_ms.iloc[0]) / 1000
    fig, ax = plt.subplots(4, 1, figsize=(10, 9), sharex=True, gridspec_kw=dict(hspace=0.35, top=0.92))

    for k, c in enumerate(["P1", "P2"]):
        ax[0].plot(t, df[c], color=SLOTS[k], label=f"{c} ({'inlet' if k == 0 else 'outlet'})")
    ax[0].set_ylabel("Pressure (kPa)")

    for k, c in enumerate(["Q1", "Q2"]):
        ax[1].plot(t, df[c], color=SLOTS[k], label=f"{c} ({'inlet' if k == 0 else 'outlet'})")
    ax[1].set_ylabel("Flow (L/min)")

    dq = (df.Q1 - df.Q2).rolling(10, center=True, min_periods=1).mean()     # 1 s moving average
    ax[2].plot(t, dq, color=INK2, label="Q1 − Q2 (1 s average)")
    ax[2].axhline(0, color=MUTED, lw=0.8)
    if r.label:
        ax[2].axhline(r.leak_lpm, color=INK2, lw=1, ls="--", label=f"jug: {r.leak_lpm:.2f} L/min")
    ax[2].set_ylabel("Flow loss (L/min)")

    for i in range(4):
        ax[3].plot(t, df[f"AE{i+1}_rms"], color=SLOTS[i], lw=1, label=f"PZT{i+1}")
    ax[3].set_yscale("log")
    ax[3].set_ylabel("AE rms (ADC counts)")
    ax[3].set_xlabel("Time (s)")

    for a in ax:
        a.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=9)
    fig.suptitle(f"Run {run_id} · {CLASS_NAMES[r.label]}" + (f" · {r.leak_lpm:.2f} L/min" if r.label else ""),
                 x=0.07, ha="left", fontsize=13)
    ax[0].set_title(r.condition, loc="left", fontsize=9, color=INK2)
    mark_simulated(fig)
    return fig

fig = plot_run(RUN)
save(fig, f"run_{RUN:04d}.png")
plt.show()

## 2 · Leak vs no-leak

`build_dataset()` is the same function `train.py` uses, so these are exactly the features the model sees.
Each dot below is **one run** (the median of its windows).

In [ ]:
X, y, groups, leak_lpm, data = build_dataset(RAW_DIR)
per_run = data.drop(columns="condition").groupby("run_id").median()
per_run["label"] = per_run.label.astype(int)
print(f"{len(data)} windows from {len(per_run)} runs")

KEY = {"dP": "Pressure drop P1 − P2 (kPa)", "dQ": "Flow loss Q1 − Q2 (L/min)",
       "dQ_rel": "Relative flow loss (dQ / Q1)", "AE_total": "Total AE rms (counts)"}
per_run.groupby("label")[list(KEY)].agg(["median", "min", "max"]).round(3).rename(index=SHORT)

In [ ]:
def strip_plot(ax, col):
    rng = np.random.default_rng(1)
    for lab in sorted(per_run.label.unique()):
        v = per_run.loc[per_run.label == lab, col]
        x = lab + rng.uniform(-0.15, 0.15, len(v))
        ax.scatter(x, v, s=40, color=CLASS_COLOR[lab], edgecolor=SURFACE, linewidth=1, zorder=3)
        ax.hlines(v.median(), lab - 0.3, lab + 0.3, color=INK, lw=2, zorder=4)
    ax.set_xticks(sorted(per_run.label.unique()), [SHORT[l] for l in sorted(per_run.label.unique())])
    ax.grid(axis="x", visible=False)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), gridspec_kw=dict(wspace=0.35))
for ax, (col, title) in zip(axes, KEY.items()):
    strip_plot(ax, col)
    ax.set_title(title, loc="left", fontsize=10)
    ax.tick_params(axis="x", labelsize=8)
fig.suptitle("Key features by class (one dot per run, black bar = median)", x=0.07, ha="left",
             fontsize=13, y=1.04)
mark_simulated(fig)
save(fig, "features_by_class.png")
plt.show()

### Which PZT hears each leak? (localisation)

Each row shows how the acoustic loudness is shared across the 4 PZTs (`AE{i}_ratio`, adds up to 1).
If localisation works, each leak row should be darkest at the PZT closest to it (new plan:
L1 → PZT1, L2 → PZT2, L3 → PZT3). The no-leak row shows the pump's background pattern.

In [ ]:
ratio_cols = [f"AE{i}_ratio" for i in range(1, 5)]
fp = per_run.groupby("label")[ratio_cols].median().rename(index=SHORT)
fp.columns = [f"PZT{i}" for i in range(1, 5)]

fig, ax = plt.subplots(figsize=(6.5, 3.6))
im = ax.imshow(fp.values, cmap="Blues", vmin=0, vmax=max(0.5, fp.values.max()), aspect="auto")
ax.set_xticks(range(4), fp.columns)
ax.set_yticks(range(len(fp)), fp.index)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
for i in range(fp.shape[0]):
    for j in range(4):
        v = fp.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10,
                color="white" if v > 0.6 * im.norm.vmax else INK)
fig.colorbar(im, ax=ax, label="Share of total AE rms", shrink=0.85)
ax.set_title("Acoustic fingerprint per class (median over runs)", loc="left", fontsize=12)
mark_simulated(fig)
save(fig, "ae_fingerprint.png")
plt.show()
fp.round(3)

### Does the flow loss match the jug-measured leak?

Each dot is one run. On the dashed line, `Q1 − Q2` equals the jug measurement exactly. A constant offset
usually means `N` (flow null) needs repeating. A slope that isn't 1 means one K-factor is off
(re-run `calibrate_flow.py`). No-leak runs should sit at (0, 0).

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for lab in sorted(per_run.label.unique()):
    d = per_run[per_run.label == lab]
    ax.scatter(d.leak_lpm, d.dQ, s=40, color=CLASS_COLOR[lab], edgecolor=SURFACE, linewidth=1,
               label=SHORT[lab], zorder=3)
lim = max(per_run.leak_lpm.max(), per_run.dQ.max()) * 1.1
ax.plot([0, lim], [0, lim], color=MUTED, ls="--", lw=1, label="perfect mass balance")
ax.set_xlim(-0.05 * lim, lim); ax.set_ylim(-0.05 * lim, lim)
ax.set_xlabel("Leak measured with jug (L/min)")
ax.set_ylabel("Flow loss Q1 − Q2 (L/min, run median)")
ax.legend(loc="upper left", fontsize=9)
ax.set_title("Flow sensors vs jug measurement", loc="left", fontsize=12)
mark_simulated(fig)
save(fig, "mass_balance.png")
plt.show()

leaks = per_run[per_run.label > 0]
if len(leaks) >= 2:
    slope, offset = np.polyfit(leaks.leak_lpm, leaks.dQ, 1)
    print(f"Leak runs: dQ = {slope:.2f} x jug {offset:+.3f} L/min  (ideal: 1.00 x + 0)")

### Which features separate leak from no-leak best?

Effect size (Cohen's d) for each feature: the difference between the leak and no-leak run medians,
divided by their spread. Above about 0.8 counts as a large difference. Unlike the random-forest
importances in `train.py`, this is one feature at a time, with no model. Treat it as a sanity check:
if a feature that physics says should matter (dQ, dP) has a small d, look at the sensor first.

In [ ]:
feat_cols = [c for c in X.columns if c in per_run.columns]
leak, normal = per_run[per_run.label > 0][feat_cols], per_run[per_run.label == 0][feat_cols]
pooled = np.sqrt((leak.var() * (len(leak) - 1) + normal.var() * (len(normal) - 1))
                 / max(len(leak) + len(normal) - 2, 1))
d = ((leak.mean() - normal.mean()) / pooled.replace(0, np.nan)).dropna()
top = d.reindex(d.abs().sort_values(ascending=False).index)[:12][::-1]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(top.index, top.abs(), color=SLOTS[0], height=0.6)
for k, (name, v) in enumerate(top.items()):
    ax.text(abs(v), k, f"  {'+' if v > 0 else '−'}{abs(v):.1f}", va="center", fontsize=8, color=INK2)
ax.axvline(0.8, color=MUTED, ls="--", lw=1)
ax.text(0.8, len(top) - 0.4, " large (0.8)", fontsize=8, color=MUTED)
ax.set_xlabel("|Cohen's d|  (sign: + = higher during leaks)")
ax.grid(axis="y", visible=False)
ax.set_title("Top 12 features: leak vs no-leak (per run)", loc="left", fontsize=12)
mark_simulated(fig)
save(fig, "effect_size.png")
plt.show()

## Notes for the real data
- **Noise runs first** (pump off; pump on with no leak at each PWM): plot each with Part 1. The no-leak
  AE level at each PWM is the floor every leak must rise above.
- **No-leak runs with transients** (speed changes, knocks, valve changes) must stay at dQ ≈ 0 in the
  mass-balance plot. If they don't, the model will raise false alarms.
- Figures from simulated data are saved with a `SIM_` prefix and a watermark. **Delete `results/` before
  recording real data** (CLAUDE.md), then re-run this notebook from the top.